In [ ]:
!pip install onnxruntime

In [ ]:
import onnxruntime
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import math

In [ ]:
test_df = pd.read_csv("/kaggle/input/digit-recognizer/test.csv")
test_X = test_df.values.astype(np.float32)

In [ ]:
ort_session = onnxruntime.InferenceSession("/kaggle/input/pytorch-lightning-onnx-export/baseline.onnx")
input_name = ort_session.get_inputs()[0].name

In [ ]:
test_pred = []
for x in tqdm(test_X):
    test_pred.extend(ort_session.run(None, {input_name: np.reshape(x, (-1, 1, 28*28))})[0])
test_pred = np.stack(test_pred)
test_pred.shape

# Batch mode

In [ ]:
batch_size = 128
test_pred = []
for batch in tqdm(np.array_split(test_X, math.ceil(len(test_X)/batch_size), axis=0)):
    test_pred.extend(ort_session.run(None, {input_name: np.reshape(batch, (-1, 1, 28*28))})[0])
test_pred = np.stack(test_pred)
test_pred.shape

In [ ]:
test_pred = np.argmax(test_pred, -1)
test_pred

In [ ]:
submission = pd.DataFrame({
    "ImageId": range(1, len(test_pred)+1),
    "Label": test_pred
})
submission.to_csv("submission.csv", index=False)
submission

# Check if sumbission is still the same

In [ ]:
submission_torch = pd.read_csv('/kaggle/input/pytorch-lightning-onnx-export/submission.csv')
(submission_torch['Label'] == submission['Label']).mean()